# IntelliOps — LSTM Anomaly Detector

An LSTM-based forecasting model for service request-rate metrics.

The runtime pipeline in this project uses **managed anomaly detection** (CloudWatch Alarms + Lookout for Metrics) to keep costs low and to avoid maintaining a hosted SageMaker endpoint. This notebook is the *reference implementation* of the alternative — a custom LSTM you'd bring in when the workload has domain-specific patterns that off-the-shelf detectors miss.

**What the model learns:** given a rolling window of the last N observed values, predict the next value. Anomalies are points where the observed value diverges sharply from the prediction (large residual).

**Why LSTM here:** request rate has strong autocorrelation (traffic follows daily cycles, promo spikes, weekend patterns). An LSTM captures that context far better than static Z-score bands.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'training on {DEVICE}')

## 2. Generate synthetic traffic

We fabricate one week of per-minute request-rate observations. The signal has:
- A daily sinusoid (peak mid-afternoon, trough overnight)
- Small Gaussian jitter
- Three injected anomalies (a spike, a sag, and a plateau)

This mirrors what a real payment or order service looks like on a Grafana dashboard.

In [ ]:
MINUTES = 7 * 24 * 60     # one week
t = np.arange(MINUTES)

# Baseline: 60 req/s average, ±30 daily oscillation
baseline = 60 + 30 * np.sin(2 * np.pi * t / (24 * 60) - np.pi / 2)

# Weekly dip on weekends (last 2 days)
weekend = np.where(t > 5 * 24 * 60, -15, 0)

noise = np.random.normal(0, 3, MINUTES)
signal = baseline + weekend + noise

# Inject anomalies at known offsets
signal[2000:2020] += 80   # spike
signal[4500:4530] -= 40   # sag
signal[7200:7300] = 90    # plateau (traffic pinned high)

df = pd.DataFrame({'minute': t, 'rate': signal})
print(df.describe().round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df['minute'] / 60, df['rate'], linewidth=0.5)
ax.set_xlabel('hours')
ax.set_ylabel('req/s')
ax.set_title('Synthetic request rate — 1 week, with 3 injected anomalies')
for start in [2000, 4500, 7200]:
    ax.axvspan(start / 60, (start + 30) / 60, color='red', alpha=0.15)
plt.tight_layout()
plt.show()

## 3. Build training windows

For each timestep `t`, the model sees the previous 60 minutes and must predict the value at `t+1`. We train on the first 5 days and validate on days 6–7.

In [ ]:
WINDOW = 60
TRAIN_MINUTES = 5 * 24 * 60

values = df['rate'].values
mean, std = values[:TRAIN_MINUTES].mean(), values[:TRAIN_MINUTES].std()
scaled = (values - mean) / std

def make_windows(series: np.ndarray, window: int):
    X, y = [], []
    for i in range(len(series) - window):
        X.append(series[i:i + window])
        y.append(series[i + window])
    return np.array(X), np.array(y)

X_train, y_train = make_windows(scaled[:TRAIN_MINUTES], WINDOW)
X_val,   y_val   = make_windows(scaled[TRAIN_MINUTES:], WINDOW)

print(f'train windows: {X_train.shape}  val windows: {X_val.shape}')

## 4. LSTM model

Two-layer LSTM with a hidden size of 32, followed by a linear head predicting one scalar. Kept deliberately small — enough capacity for the pattern, cheap to train and to host.

In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

MODEL_CONFIG = {'input_size': 1, 'hidden_size': 32, 'num_layers': 2}
model = LSTMForecaster(**MODEL_CONFIG).to(DEVICE)
print(model)

## 5. Train

In [ ]:
EPOCHS = 20
BATCH  = 128

def to_loader(X, y, batch, shuffle):
    tX = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)
    ty = torch.tensor(y, dtype=torch.float32).unsqueeze(-1)
    return DataLoader(TensorDataset(tX, ty), batch_size=batch, shuffle=shuffle)

train_loader = to_loader(X_train, y_train, BATCH, shuffle=True)
val_loader   = to_loader(X_val,   y_val,   BATCH, shuffle=False)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

history = []
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)
    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            val_loss += criterion(model(xb), yb).item() * xb.size(0)
    val_loss /= len(val_loader.dataset)

    history.append((train_loss, val_loss))
    print(f'epoch {epoch+1:02d}  train={train_loss:.4f}  val={val_loss:.4f}')

## 6. Evaluate — residual-based anomaly detection

Once trained, we score the whole series. A point is flagged as anomalous when its residual (|actual − predicted|) exceeds 3× the training-set residual standard deviation.

In [ ]:
model.eval()
X_all, _ = make_windows(scaled, WINDOW)
with torch.no_grad():
    preds_scaled = model(torch.tensor(X_all, dtype=torch.float32).unsqueeze(-1).to(DEVICE)).cpu().numpy().flatten()

preds = preds_scaled * std + mean
actuals = values[WINDOW:]
residuals = np.abs(actuals - preds)

# Threshold from train residuals only
train_residuals = residuals[:TRAIN_MINUTES - WINDOW]
residual_threshold = float(train_residuals.mean() + 3 * train_residuals.std())
print(f'residual threshold: {residual_threshold:.2f}')

anomalies = np.where(residuals > residual_threshold)[0]
print(f'anomalies flagged: {len(anomalies)}')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(actuals, label='actual', linewidth=0.4)
ax.plot(preds,   label='predicted', linewidth=0.4, alpha=0.7)
ax.scatter(anomalies, actuals[anomalies], color='red', s=6, label='anomaly', zorder=5)
ax.legend()
ax.set_title('LSTM predictions vs actual, with flagged anomalies')
plt.tight_layout()
plt.show()

## 7. Save checkpoint

The checkpoint is a single `.pt` file containing weights, scaler stats, and the residual threshold — everything `inference.py` needs to reconstruct the model and score new points.

To promote this to a SageMaker endpoint, tar the `.pt` file with an `inference.py` entrypoint and upload to S3, then create a `sagemaker.pytorch.PyTorchModel` and deploy. That endpoint costs ~$72/month at the smallest instance size, which is why we prefer the managed-detector runtime path unless custom modelling is genuinely needed.

In [ ]:
checkpoint = {
    'state_dict': model.state_dict(),
    'model_config': MODEL_CONFIG,
    'scaler': {'mean': float(mean), 'std': float(std)},
    'residual_threshold': residual_threshold,
    'window': WINDOW,
}
torch.save(checkpoint, 'model.pt')
print('saved model.pt — hand this to inference.py or ship to S3 for SageMaker.')